In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent.parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [ ]:
N = CONFIG['n_sim']
rng = np.random.default_rng(SEED)
TAG = 'lognormal_grw_s'
GENERATED_FROM = 1

In [ ]:
params = pd.read_csv(INPUT / TAG / 'parameters.csv')
panel = pd.read_csv(INPUT / 'data/panel.csv', index_col='dblp_id')
DEST = OUTPUT / TAG
DEST.mkdir(exist_ok=True)
sim = np.zeros((Y+1, N))

In [ ]:
# same initialization and zero floor as pooled GRW
sim[0] = rng.choice(panel['0'].dropna().to_numpy(), N, replace=True)
z = np.log(sim[0]+EPS)
for row in params.itertuples():
    for t in range(row.start, row.end+1):
        z_next = z + rng.normal(row.mu, row.sigma, N)
        sim[t+1] = np.maximum(np.exp(z_next)-EPS, 0)
        z = np.log(sim[t+1]+EPS)

In [ ]:
np.save(DEST / 'trajectories.npy', sim, allow_pickle=False)
pd.DataFrame([{'model':TAG, 'seed':SEED, 'n_sim':N, 'n_years':Y+1, 'generated_from_age':GENERATED_FROM,
               'orientation':'age x scholar', 'minimum':sim.min(), 'maximum':sim.max(),
               'finite':np.isfinite(sim).all()}]).to_csv(DEST / 'metadata.csv', index=False)
display(pd.read_csv(DEST / 'metadata.csv'))